# Grid search analysis

Lecture et visualisation d'un grid_summary CSV produit par `scripts/grid_aggregate.py`.

**Workflow :**
1. Lance `python scripts/grid_search.py --grid scripts/grid_example.json`
2. Lance `python scripts/grid_aggregate.py --manifest outputs/grid_manifest_<ts>.json`
3. Ouvre ce notebook et pointe `GRID_CSV` sur le fichier produit.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

OUTPUTS = Path('..').resolve() / 'outputs'

# Pick the latest grid_summary_*.csv by default.
candidates = sorted(OUTPUTS.glob('grid_summary_*.csv'))
assert candidates, 'No grid_summary_*.csv found — run scripts/grid_aggregate.py first.'
GRID_CSV = candidates[-1]
print('Using', GRID_CSV)

df = pd.read_csv(GRID_CSV)
df.head()

## Quelles colonnes sont des hyperparamètres ?

Les colonnes `hedging_agent.*`, `run.*`, `training_schedule.*`, `simulation.*` dont il y a plus d'une valeur unique.

In [ ]:
META = {'run_id','timestamp','process','agent','benchmark','seed','train_episodes','eval_episodes'}
METRICS = {'mean_rl','std_rl','skew_rl','cvar95_rl','cvar99_rl','y_rl','mean_bm','std_bm','skew_bm','cvar95_bm','cvar99_bm','y_bm','improvement_pct','ok'}
grid_cols = [c for c in df.columns if c not in META and c not in METRICS and df[c].nunique(dropna=False) > 1]
print('Grid columns (varying):', grid_cols)

## Agréger par hyperparams (moyenne/std sur les seeds)

In [ ]:
group_cols = [c for c in grid_cols if c != 'run.seed']
agg = (
    df[df['ok']]
    .groupby(group_cols, dropna=False)
    .agg(
        n_seeds=('seed','count'),
        mean_y_rl=('y_rl','mean'),
        std_y_rl=('y_rl','std'),
        mean_improv=('improvement_pct','mean'),
        std_improv=('improvement_pct','std'),
        mean_mean_rl=('mean_rl','mean'),
        mean_std_rl=('std_rl','mean'),
        mean_cvar95_rl=('cvar95_rl','mean'),
        mean_cvar99_rl=('cvar99_rl','mean'),
    )
    .reset_index()
    .sort_values('mean_improv', ascending=False)
)
agg.head(20)

## Heatmap 2D (moyenne improvement_pct sur les seeds)

Choisis 2 axes parmi `grid_cols` (les autres sont agrégés).

In [ ]:
non_seed = [c for c in grid_cols if c != 'run.seed']
if len(non_seed) >= 2:
    X_AXIS = non_seed[0]
    Y_AXIS = non_seed[1]
    pivot = (
        df[df['ok']]
        .groupby([Y_AXIS, X_AXIS])['improvement_pct']
        .mean()
        .unstack(X_AXIS)
    )
    fig, ax = plt.subplots(figsize=(8,5))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0, cbar_kws={'label':'Improvement %'}, ax=ax)
    ax.set_title(f'Mean improvement % vs benchmark (averaged over seeds, other hyperparams)')
    plt.tight_layout()
    plt.show()
else:
    print('Pas assez d\u2019axes varier pour une heatmap 2D.')

## Scatter : y_rl vs CVaR(95) RL, coloré par hyperparam clé

In [ ]:
if grid_cols:
    color_by = grid_cols[0]  # change pour voir un autre hyperparam
    ok = df[df['ok']]
    fig, ax = plt.subplots(figsize=(8,5))
    sc = ax.scatter(ok['cvar95_rl'], ok['y_rl'], c=ok[color_by], cmap='viridis', s=40, edgecolor='white')
    if len(ok['y_bm']):
        ax.axhline(ok['y_bm'].iloc[0], color='red', linestyle='--', alpha=0.5, label=f"y_bm = {ok['y_bm'].iloc[0]:.1f}")
    if len(ok['cvar95_bm']):
        ax.axvline(ok['cvar95_bm'].iloc[0], color='red', linestyle=':', alpha=0.4, label=f"cvar95_bm = {ok['cvar95_bm'].iloc[0]:.1f}")
    ax.set_xlabel('CVaR(95) RL (% option price)')
    ax.set_ylabel('Y objective RL (% option price)')
    ax.set_title(f'Y vs CVaR(95) — colored by {color_by}')
    ax.legend()
    plt.colorbar(sc, ax=ax, label=color_by)
    plt.tight_layout()
    plt.show()

## Top 5 meilleurs runs (par improvement moyen sur les seeds)

In [ ]:
top = agg.head(5)
print('TOP 5 :')
print(top.to_string(index=False))

# Runs concrets derrière le top-1 (utile pour les ouvrir dans plot_run)
best = top.iloc[0]
mask = np.ones(len(df), dtype=bool)
for c in group_cols:
    mask &= df[c] == best[c]
print('\nRun IDs du top-1 :')
print(df[mask][['run_id','seed','y_rl','improvement_pct']].to_string(index=False))

## Bar chart de sensibilité (1D) pour chaque hyperparam

In [ ]:
for col in grid_cols:
    if col == 'run.seed':
        continue
    fig, ax = plt.subplots(figsize=(6,3))
    sub = df[df['ok']].groupby(col)['improvement_pct'].agg(['mean','std']).reset_index()
    ax.bar(range(len(sub)), sub['mean'], yerr=sub['std'], capsize=4, color='#4A90E2', edgecolor='white')
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels([str(x) for x in sub[col]])
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_title(f'Sensibilit\u00e9 de improvement_pct \u00e0 {col}')
    ax.set_ylabel('Improvement % (mean \u00b1 std over other hyperparams)')
    plt.tight_layout()
    plt.show()